# Simple: Numeric Generalization with PAMOLA.CORE

**Goal**: Learn numeric generalization basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply binning generalization using execute()
- Compare before/after results
- Understand privacy-utility tradeoffs

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/generalization/
        └── 01_numeric_generalization_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.generalization.numeric_op import (
        NumericGeneralizationOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv

    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/sample.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with records including numeric fields like age, salary, score

In [ ]:
# Define path to sample data (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

# Check if file exists
if not data_path.exists():
    print(f"⚠️  File not found: {data_path}")
    print("Creating sample data...\n")
    
    # Create directory
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with realistic numeric values
    np.random.seed(42)
    sample_data = pd.DataFrame({
        'record_id': range(1, 101),
        'age': np.random.randint(18, 75, 100),
        'salary': np.random.randint(30000, 150000, 100),
        'score': np.round(np.random.uniform(50, 100, 100), 2),
        'experience_years': np.random.randint(0, 30, 100),
        'department': np.random.choice(['Sales', 'Engineering', 'HR', 'Marketing', 'Finance'], 100)
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created at: {data_path}\n")

# Load data using pandas
df = read_csv(data_path)

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}, mean={df[col].mean():.2f}")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure field name** for generalization

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "years_of_experience"  # Customize this!
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "years_of_experience"  # Field to generalize - CUSTOMIZE THIS!
    }

# Get config
kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists in dataset
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

# Validate field is numeric
if not pd.api.types.is_numeric_dtype(df[field_name]):
    raise ValueError(
        f"❌ Field '{field_name}' is not numeric (dtype: {df[field_name].dtype})!\n"
        f"Numeric generalization requires numeric fields.\n"
        f"Available numeric columns: {[c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]}"
    )

print(f"✓ Field to generalize: '{field_name}'")
print(f"  Data type: {df[field_name].dtype}")
print(f"  Range: {df[field_name].min()} - {df[field_name].max()}")
print(f"  Mean: {df[field_name].mean():.2f}")
print(f"  Std Dev: {df[field_name].std():.2f}")
print(f"  Unique values: {df[field_name].nunique()}")

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_numeric_001",
    task_type="numeric_generalization",
    description=f"Numeric generalization of '{field_name}' field using binning strategy.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config kwargs (without field_name for execute)
execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create NumericGeneralizationOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `field_name='{field_name}'`: Field to generalize
- `strategy='binning'`: Group values into bins
- `bin_count=5`: Create 5 bins (e.g., 18-30, 31-43, ...)
- `binning_method='equal_width'`: Equal-width bins
- `mode='REPLACE'`: Modify original field (or 'ENRICH' for new field)
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = NumericGeneralizationOperation(
    # Core parameters
    field_name=field_name,               # From config
    mode='REPLACE',
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Strategy parameters
    strategy='binning',                  # Options: 'binning', 'rounding', 'range'
    bin_count=5,                         # Number of bins for binning strategy
    binning_method='equal_width',        # Options: 'equal_width', 'equal_frequency', 'quantile'
    precision=0,                         # For rounding strategy (not used here)
    range_limits=None,                   # For range strategy (not used here)
    
    # Privacy parameters
    quasi_identifiers=None,              # Optional: list of QI fields for k-anonymity
    ka_risk_field=None,                  # Optional: pre-calculated k-anonymity risk field
    risk_threshold=5,                    # Minimum k-anonymity threshold
    vulnerable_record_strategy='suppress',  # How to handle vulnerable records
    
    # Processing settings
    use_vectorization=False,             # Disable for small data
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,         # Create visualization artifacts
    save_output=True,                    # Save results to files
    force_recalculation=False            # Use cache when available
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Strategy:         {operation.strategy}")
print(f"  Bin count:        {operation.bin_count}")
print(f"  Binning method:   {operation.binning_method}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing numeric generalization...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the generalized output from task_dir
- Compare with original data
- Review metrics and artifacts

**Generated artifacts:**
- Generalized CSV in output/
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Generalized Records:")
    display_cols = [field_name]
    if 'record_id' in result_df.columns:
        display_cols.insert(0, 'record_id')
    if 'department' in result_df.columns:
        display_cols.append('department')
    print(result_df[display_cols].head(10))
    
    # Distribution comparison
    print(f"\n📈 {field_name.title()} Distribution (After Generalization):")
    dist = result_df[field_name].value_counts().sort_index()
    for bin_range, count in dist.items():
        pct = (count / len(result_df)) * 100
        bar = '█' * int(pct / 2)
        print(f"  {str(bin_range):20s} {bar:30s} {count:3d} ({pct:5.1f}%)")
    
    # Calculate information loss metrics
    original_unique = df[field_name].nunique()
    generalized_unique = result_df[field_name].nunique()
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:       {len(df)}")
    print(f"  Generalized records:    {len(result_df)}")
    print(f"  Original unique values: {original_unique}")
    print(f"  Generalized bins:       {generalized_unique}")
    print(f"  Information reduction:  {(1 - generalized_unique/original_unique)*100:.1f}%")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Fewer unique values = Better privacy protection with controlled information loss!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Generalized CSV
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

# List all subdirectories and files
for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:  # Show first 5 files
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")
        if len(files) > 5:
            print(f"   ... and {len(files) - 5} more files")
        print()

print("=" * 80)
print("\n✅ All artifacts saved successfully!")
print(f"\n💡 TIP: Navigate to {task_dir.absolute()} to inspect files manually")

## Step 7: Detailed Artifact Review

**How to use:**
- Review metrics JSON with detailed statistics
- Inspect output data distribution
- View visualizations

**Note:** Always shows the NEWEST files from the latest operation run.

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. Display Metrics JSON (NEWEST FILE)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types inside IF
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            # Use only filtered files
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            # Fallback: nothing left after filtering → use all
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # Display metadata
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display effectiveness metrics
            if 'effectiveness' in metrics:
                print("\n📈 EFFECTIVENESS:")
                eff = metrics['effectiveness']
                print(f"   Total Records: {eff.get('total_records', 'N/A'):,}")
                print(f"   Original Unique: {eff.get('original_unique', 'N/A')}")
                print(f"   Anonymized Unique: {eff.get('anonymized_unique', 'N/A')}")
                print(f"   Effectiveness Ratio: {eff.get('effectiveness_ratio', 0):.4f}")
                print(f"   Null Increase: {eff.get('null_increase', 0):.4f}")
            
            # Display performance metrics
            if 'performance' in metrics:
                print("\n⚡ PERFORMANCE:")
                perf = metrics['performance']
                print(f"   Duration: {perf.get('duration_seconds', 0):.4f}s")
                print(f"   Records Processed: {perf.get('records_processed', 0):,}")
                print(f"   Records/Second: {perf.get('records_per_second', 0):.2f}")
            
            # Display field info (original vs processed)
            if 'field_info' in metrics:
                print("\n📊 FIELD COMPARISON:")
                field_info = metrics['field_info']
                
                # Original
                if 'original' in field_info:
                    orig = field_info['original']
                    print(f"\n   ORIGINAL:")
                    print(f"      Total Count: {orig.get('total_count', 0):,}")
                    print(f"      Unique Count: {orig.get('unique_count', 0)}")
                    print(f"      Mean: {orig.get('mean', 0):.2f}")
                    print(f"      Std Dev: {orig.get('std', 0):.2f}")
                    print(f"      Min: {orig.get('min', 0)}")
                    print(f"      Max: {orig.get('max', 0)}")
                    print(f"      Null Count: {orig.get('null_count', 0)}")
                
                # Processed
                if 'processed' in field_info:
                    proc = field_info['processed']
                    print(f"\n   PROCESSED (Generalized):")
                    print(f"      Total Count: {proc.get('total_count', 0):,}")
                    print(f"      Unique Bins: {proc.get('unique_count', 0)}")
                    print(f"      Null Count: {proc.get('null_count', 0)}")
                    
                    if 'top_values' in proc and proc['top_values']:
                        print(f"\n      Bin Distribution:")
                        for val, count in list(proc['top_values'].items())[:10]:
                            print(f"         {val:<20}: {count:>3}")
            
            # Display generalization info
            if 'generalization' in metrics:
                print("\n🔄 GENERALIZATION:")
                gen = metrics['generalization']
                print(f"   Strategy: {gen.get('strategy', 'N/A')}")
                print(f"   Reduction Ratio: {gen.get('reduction_ratio', 0):.2%}")
                
                if 'parameters' in gen:
                    params = gen['parameters']
                    print(f"\n   Parameters:")
                    print(f"      Bin Count: {params.get('bin_count', 'N/A')}")
                    print(f"      Binning Method: {params.get('binning_method', 'N/A')}")
                    print(f"      Precision: {params.get('precision', 'N/A')}")
            
            # Display information loss metrics
            if 'numeric_info_loss' in metrics:
                print("\n📉 INFORMATION LOSS:")
                info_loss = metrics['numeric_info_loss']
                print(f"   Precision Loss: {info_loss.get('precision_loss', 0):.2%}")
                print(f"   Range Reduction: {info_loss.get('range_reduction', 0):.2%}")
                print(f"   Value Reduction: {info_loss.get('value_reduction_ratio', 0):.2%}")
                print(f"   Avg Bin Size: {info_loss.get('average_bin_size', 0):.2f}")
            
            # Display privacy metrics
            if 'privacy_metrics' in metrics:
                print("\n🔒 PRIVACY METRICS:")
                privacy = metrics['privacy_metrics']
                status = privacy.get('status', 'N/A')
                print(f"   Status: {status}")
                if status == 'SKIPPED':
                    print(f"   Reason: {privacy.get('reason', 'N/A')}")
                print(f"   Summary: {metrics.get('privacy_summary', 'N/A')}")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. Display Output Data Sample (NEWEST FILE)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df.head(10).to_string()}")
            
            # Show value counts for the generalized field
            if field_name in output_df.columns:
                print(f"\n\n📊 {field_name.title()} Bin Distribution:")
                print("-" * 80)
                dist = output_df[field_name].value_counts().sort_index()
                total = len(output_df)
                print(f"{'Bin Range':<30} {'Count':>6} {'Percentage':>12}")
                print("-" * 50)
                for bin_range, count in dist.items():
                    pct = (count / total) * 100
                    print(f"{str(bin_range):<30} {count:>6} {pct:>11.2f}%")
            else:
                print(f"\n⚠️  Field '{field_name}' not found in output")
                print(f"Available columns: {', '.join(output_df.columns)}")
        
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. Display Visualizations (NEWEST FILES ONLY - matching latest operation)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                name_parts = viz_file.stem.split('_')
                viz_name = viz_file.stem.replace('_', ' ').replace('anonymization numeric', '').strip().title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV from examples/data_examples/  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with field config  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review metrics and artifacts  

**Key takeaways:**
- `operation.execute()` handles end-to-end processing
- Execution behavior (visualizations, output saving, caching) configured in operation constructor
- Field name is configurable via `create_config_kwargs()`
- Binning strategy groups numeric values into discrete intervals
- Fewer unique values = better privacy with controlled information loss
- All artifacts saved in structured directories

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_numeric_generalization_advanced.ipynb`** includes:
- All 3 strategies (binning, rounding, range-based)
- Custom binning methods (equal_width, equal_frequency, quantile)
- Multi-field processing pipelines
- Advanced privacy metrics (k-anonymity integration)
- Performance optimization with vectorization
- Risk-based processing with vulnerable record handling

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Privacy Metrics Guide](../../docs/privacy_metrics.md)
- 📊 [Numeric Generalization Strategies](../../docs/generalization_strategies.md)